In [ ]:
import sys

package_path = ".."
if package_path not in sys.path:
    sys.path.append(package_path)

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt

from QoraFlow.config import ExpConfig

In [ ]:
def get_hdf5_images(file: str, config: ExpConfig) -> npt.NDArray:
    """get images form hdf5"""
    with h5py.File(file, "r") as hf:
        if "detector" not in hf:
            raise KeyError(f"Key 'detector' not found in {file}")

        images = np.asarray(
            hf[
                f"detector/{config.param.hutch.value}/{config.param.detector.value}/image/block0_values"
            ]
        )

        return images  # np.maximum(images, 0)

In [ ]:
from QoraFlow.filesystem import get_run_scan_directory

config = load_config(reload=True)
file = get_run_scan_directory(config.path.load_dir, 18, 1, 53)
print(file)
arr = get_hdf5_images(file, config)

In [ ]:
np.shape(arr)

In [ ]:
indbad = np.where(arr < 0)

In [ ]:
indlist = []
listmin = []

for ii in range(np.shape(arr)[0]):
    indlist.append(np.where(arr[ii, :, :] < -17500))

    listmin.append(arr[ii, :, :].min())

In [ ]:
one_zeros = np.where(arr < -10000, 1, 0)
mask = np.nansum(one_zeros, 0)
mask[mask == 0] = 1
mask[mask == np.float32(np.shape(arr)[0])] = 0

In [ ]:
masked_arr = arr * mask
masked_arr_sum = np.nansum(masked_arr, 0)

In [ ]:
np.where(masked_arr_sum < 500)

In [ ]:
fig1 = plt.figure(figsize=(10, 10))
ax = fig1.add_subplot(131)
arr[arr < 0] = 0
to_plot = np.nansum(arr, 0)
to_plot[11, 25] = 0
to_plot = np.transpose(to_plot)
img1 = ax.imshow(to_plot, interpolation="none")
plt.colorbar(img1)

ax2 = fig1.add_subplot(132)
masked_arr_sum[masked_arr_sum < 0] = 0
img2 = ax2.imshow(np.transpose(masked_arr_sum), interpolation="none")
plt.colorbar(img2)


ax3 = fig1.add_subplot(133)
img3 = ax3.imshow(to_plot - np.transpose(masked_arr_sum), interpolation="none")
plt.colorbar(img3)

In [ ]:
def h5_tree(val, pre="", out=""):
    length = len(val)
    for key, val in val.items():
        length -= 1
        if length == 0:  # the last item
            if type(val) == h5py._hl.group.Group:
                out += pre + "└── " + key + "\n"
                out = h5_tree(val, pre + "    ", out)
            else:
                out += pre + "└── " + key + f" {val.shape}\n"
        else:
            if type(val) == h5py._hl.group.Group:
                out += pre + "├── " + key + "\n"
                out = h5_tree(val, pre + "│   ", out)
            else:
                out += pre + "├── " + key + f" {val.shape}\n"
    return out


testh5 = config.path.load_dir + "run=018/scan=001/p0053.h5"
filename = testh5
with h5py.File(filename, "r") as file:
    structure = h5_tree(file)
    print(structure)

In [ ]:
filename = testh5

with h5py.File(filename, "r") as f:
    detimage = f["/detector/eh1/jungfrau2/image/block0_values"][:, :, :]

In [ ]:
import tifffile

fn_save = "data/example/analysis/test_DS1/img1200_18_53.tif"
tifffile.imwrite(fn_save, detimage)

In [ ]:
np.shape(detimage)

fig1 = plt.figure(figsize=(10, 10))
ax = fig1.add_subplot(111)
detimage[detimage < 0] = 0
to_plot = np.nansum(detimage[:, :, :], 0)
to_plot[to_plot > 1600] = 0
to_plot[11, 25] = 0
to_plot = to_plot
img1 = ax.imshow(to_plot, interpolation="none")
plt.colorbar(img1)

In [ ]:
filename = testh5

with h5py.File(filename, "r") as f:
    block0_items = list(f["/metadata/block0_items"])
    block1_items = list(f["/metadata/block1_items"])
    block2_items = list(f["/metadata/block2_items"])
    block3_items = list(f["/metadata/block3_items"])
    block0_values = list(f["/metadata/block0_values"])

with h5py.File(filename, "r") as f:
    # Print all root level object names (aka keys)
    # these can be group or dataset names
    print("Keys: {}".format(f["/metadata/"].keys()))
    # get first object name/key; may or may NOT be a group
    a_group_key = next(iter(f.keys()))

    # get the object type for a_group_key: usually group or dataset
    print(type(f[a_group_key]))

    # If a_group_key is a group name,
    # this gets the object names in the group and returns as a list
    data = list(f[a_group_key])

    # If a_group_key is a dataset name,
    # this gets the dataset values and returns as a list
    data = list(f[a_group_key])
    # # preferred methods to get dataset values:
    # ds_obj = f[a_group_key]      # returns as a h5py dataset object
    # ds_arr = f[a_group_key][()]  # returns as a numpy array

In [ ]:
def print_list(ll):
    for ii in range(len(ll)):
        print(f"{ii: } {ll[ii]}")


print("block 0")
print_list(block0_items)
print("block 1")
print_list(block1_items)
print("block 2")
print_list(block2_items)
print("block 3")
print_list(block3_items)

In [ ]:
print(np.where(to_plot == to_plot.max()))
print(np.where(masked_arr_sum == masked_arr_sum.max()))

In [ ]:
np.ravel_multi_index([np.argmax(np.nansum(arr, 0))], arr.shape[1:])

In [ ]:
test = set(np.reshape(one_zeros_sum, -1))
test

In [ ]:
fig1 = plt.figure(figsize=(10, 10))
ax = fig1.add_subplot(111)
ax.plot(listmin, "-o")

In [ ]:
indlist[0][1][0:]

In [ ]:
fig1 = plt.figure(figsize=(10, 10))
ax = fig1.add_subplot(111)
ax.scatter(indlist[0][0], indlist[0][1], 30)
ax.scatter(indlist[400][0], indlist[400][1], 10)
ax.set_aspect(1)
# ax.plot(indlist[100][0], '-o')0

In [ ]:
%matplotlib widget

In [ ]:
fig1 = plt.figure(figsize=(10, 10))

mask = np.ones((np.shape(arr)[1], np.shape(arr)[2]))
mask[:, :] = arr[0, :, :]
mask[mask < -17500] = np.nan
mask[~np.isnan(mask)] = 1


ax = fig1.add_subplot(211)
img1 = ax.imshow(arr[0, :, :] * mask, interpolation="none")
plt.colorbar(img1)


ax = fig1.add_subplot(212)

img1 = ax.imshow(arr[100, :, :] * mask, interpolation="none")
plt.colorbar(img1)

In [ ]:
fig1 = plt.figure(figsize=(10, 10))
ax = fig1.add_subplot(111)
img1 = ax.imshow(np.nansum(arr, 0) * mask, interpolation="none")
plt.colorbar(img1)

In [ ]:
fig1 = plt.figure(figsize=(10, 10))
ax = fig1.add_subplot(111)
img1 = ax.imshow(arr[100, :, :], interpolation="none")
plt.colorbar(img1)